# Milestone 2: Movie Data Acquisition

*Cleaning/Formatting Flat File Source* 

Perform at least 5 data transformation and/or cleansing steps to your flat file data. The below examples are not required - they are just potential transformations you could do. If your data doesn't work for these scenarios, complete different transformations. You can do the same transformation multiple times if needed to clean your data. The goal is a clean dataset at the end of the milestone. 

 

**Examples:** 

- Replace Headers. 
- Format data int a more readable format. 
- Identify outliers and bad data. 
- Find duplicates. 
- Fix casing or inconsistent values. 
- Conduct Fuzzy Matching. 

 

Make sure you clearly number and label each transformation step (Step #1, Step #2, etc.) in your code and describe what it is doing in 1-2 sentences. 

In [1]:
# Load the necessary libraries.
import pandas as pd
import numpy as np
from scipy import stats

## I'm working with two different .csv files from the DataSet I mentioned on Milestone 1.

In [2]:
# Load .csv files containing critic reviews and movies information into pandas DataFrames for analysis.
critic_reviews_df = pd.read_csv("rotten_tomatoes_critic_reviews.csv")
movies_df = pd.read_csv("rotten_tomatoes_movies.csv")

# Critics Review DataFrame.

In [3]:
# Display first few rows of critic reviews DataFrame.
critic_reviews_df.head()

,rotten_tomatoes_link,critic_name,top_critic,publisher_name,review_type,review_score,review_date,review_content
0,m/0814255,Andrew L. Urban,False,Urban Cinefile,Fresh,NaN,2010-02-06,A fantasy adventure that fuses Greek mythology...
1,m/0814255,Louise Keller,False,Urban Cinefile,Fresh,NaN,2010-02-06,"Uma Thurman as Medusa, the gorgon with a coiff..."
2,m/0814255,NaN,False,FILMINK (Australia),Fresh,NaN,2010-02-09,With a top-notch cast and dazzling special eff...
3,m/0814255,Ben McEachen,False,Sunday Mail (Australia),Fresh,3.5/5,2010-02-09,Whether audiences will get behind The Lightnin...
4,m/0814255,Ethan Alter,True,Hollywood Reporter,Rotten,NaN,2010-02-10,What's really lacking in The Lightning Thief i...


In [4]:
# Get the original dimensions of the critic reviews DataFrame.
critic_reviews_df.shape

(1130017, 8)

In [5]:
# Display the data types of each column in the DataFrame to understand the structure of the data.
critic_reviews_df.dtypes

rotten_tomatoes_link    object
critic_name             object
top_critic                bool
publisher_name          object
review_type             object
review_score            object
review_date             object
review_content          object
dtype: object

In [6]:
# STEP 1.
# Create a new DataFrame cleaned_critic_reviews_df from critic_reviews_df
cleaned_critic_reviews_df = critic_reviews_df.copy()

In [7]:
# STEP 2.
# Format Headers: Removing underscores and capitalize the first letter of each word.
cleaned_critic_reviews_df.columns = cleaned_critic_reviews_df.columns.str.replace("_", " ").str.title()

In [8]:
# Display first few rows of cleaned_critic_reviews_df to verify change to headers.
cleaned_critic_reviews_df.head()

,Rotten Tomatoes Link,Critic Name,Top Critic,Publisher Name,Review Type,Review Score,Review Date,Review Content
0,m/0814255,Andrew L. Urban,False,Urban Cinefile,Fresh,NaN,2010-02-06,A fantasy adventure that fuses Greek mythology...
1,m/0814255,Louise Keller,False,Urban Cinefile,Fresh,NaN,2010-02-06,"Uma Thurman as Medusa, the gorgon with a coiff..."
2,m/0814255,NaN,False,FILMINK (Australia),Fresh,NaN,2010-02-09,With a top-notch cast and dazzling special eff...
3,m/0814255,Ben McEachen,False,Sunday Mail (Australia),Fresh,3.5/5,2010-02-09,Whether audiences will get behind The Lightnin...
4,m/0814255,Ethan Alter,True,Hollywood Reporter,Rotten,NaN,2010-02-10,What's really lacking in The Lightning Thief i...


In [9]:
# Calculate the total number of missing values in each column of the cleaned_critic_reviews_df.
missing_values = cleaned_critic_reviews_df.isnull().sum()
print(missing_values)

Rotten Tomatoes Link         0
Critic Name              18529
Top Critic                   0
Publisher Name               0
Review Type                  0
Review Score            305936
Review Date                  0
Review Content           65806
dtype: int64


In [10]:
# STEP 3.
# Fill missing values in "Critic Name" with the placeholder "Unknown Critic."
cleaned_critic_reviews_df["Critic Name"] = cleaned_critic_reviews_df["Critic Name"].fillna("Unknown Critic")

In [11]:
# STEP 4. 
# Define a function to clean the Review Scores and create a new column named Numeric Review Score.
def clean_review_score(value):
    if pd.isna(value) or value == "":
        return np.nan  # Keep empty cells as NaN.
    elif "/" in value:
        return float(value.split("/")[0])  # Extract numeric value before "/".
    elif isinstance(value, str) and value.replace("-", "").isalpha():  # Check for grades.
        return np.nan  # Treat grades as NaN.
    else:
        return np.nan  # Default case for unexpected formats.

# Apply "clean_review_score" function to "Review Score" column and store the cleaned numeric values in the "Numeric Review Score" column.
cleaned_critic_reviews_df["Numeric Review Score"] = cleaned_critic_reviews_df["Review Score"].apply(clean_review_score)

# Calculate the mean score, ignoring NaN values.
mean_review_score = cleaned_critic_reviews_df["Numeric Review Score"].mean()

# Fill NaN values in the new column with the mean score
cleaned_critic_reviews_df["Numeric Review Score"] = cleaned_critic_reviews_df["Numeric Review Score"].fillna(mean_review_score)

# Display "Mean Review Score".
print("Mean Review Score: ", mean_review_score)

Mean Review Score:  3.9343797641805165


In [12]:
# STEP 5.
# Fill missing values in Review Content with N/A.
cleaned_critic_reviews_df["Review Content"] = cleaned_critic_reviews_df["Review Content"].fillna("N/A")

In [13]:
# Display the total number of missing values in each column of the DataFrame after cleaning.
# "Numeric Review Score" column is a cleaned version of "Review Score" column. 
missing_values = cleaned_critic_reviews_df.isnull().sum()
print(missing_values)

Rotten Tomatoes Link         0
Critic Name                  0
Top Critic                   0
Publisher Name               0
Review Type                  0
Review Score            305936
Review Date                  0
Review Content               0
Numeric Review Score         0
dtype: int64


In [14]:
# STEP 6.
# Convert "Review Date" from string/object to datetime format.
cleaned_critic_reviews_df["Review Date"] = pd.to_datetime(cleaned_critic_reviews_df["Review Date"], errors = "coerce")

In [15]:
# Check unique values from the Review Type column to identify distinct review categories.
unique_values = cleaned_critic_reviews_df["Review Type"].unique()
print("Unique values in Review Type: ", unique_values)

Unique values in Review Type:  ['Fresh' 'Rotten']


In [16]:
# STEP 7.
# Check for duplicate reviews based on "Rotten Tomatoes Link" and "Critic Name."
duplicate_reviews = cleaned_critic_reviews_df.duplicated(subset = ["Rotten Tomatoes Link", "Critic Name"], keep = False)

# # Display duplicate rows, if there are any.
duplicate_entries = cleaned_critic_reviews_df[duplicate_reviews]
print(duplicate_entries)

# Remove duplicates, keeping the first occurrence.
cleaned_critic_reviews_df.drop_duplicates(subset = ["Rotten Tomatoes Link", "Critic Name"], inplace = True)

           Rotten Tomatoes Link       Critic Name  Top Critic  \
2                     m/0814255    Unknown Critic       False   
80                    m/0814255    Unknown Critic       False   
101                   m/0814255    Unknown Critic        True   
320      m/1000013-12_angry_men    Unknown Critic       False   
348      m/1000013-12_angry_men    Unknown Critic       False   
...                         ...               ...         ...   
1129695              m/zootopia  Jonathan Sánchez       False   
1129696              m/zootopia  Jonathan Sánchez       False   
1129987                  m/zulu    Unknown Critic       False   
1130005                  m/zulu    Unknown Critic       False   
1130006                  m/zulu    Unknown Critic       False   

              Publisher Name Review Type Review Score Review Date  \
2        FILMINK (Australia)       Fresh          NaN  2010-02-09   
80             National Post      Rotten        2.5/4  2010-02-12   
101         

In [17]:
# Get the shape of cleaned_critic_reviews_df.
cleaned_critic_reviews_df.shape

(996470, 9)

In [18]:
# Display cleaned_critic_reviews_df types of each column in the DataFrame.
cleaned_critic_reviews_df.dtypes

Rotten Tomatoes Link            object
Critic Name                     object
Top Critic                        bool
Publisher Name                  object
Review Type                     object
Review Score                    object
Review Date             datetime64[ns]
Review Content                  object
Numeric Review Score           float64
dtype: object

In [19]:
# Display first few rows of the cleaned_critic_reviews_df DataFrame.
cleaned_critic_reviews_df.head()

,Rotten Tomatoes Link,Critic Name,Top Critic,Publisher Name,Review Type,Review Score,Review Date,Review Content,Numeric Review Score
0,m/0814255,Andrew L. Urban,False,Urban Cinefile,Fresh,NaN,2010-02-06,A fantasy adventure that fuses Greek mythology...,3.93438
1,m/0814255,Louise Keller,False,Urban Cinefile,Fresh,NaN,2010-02-06,"Uma Thurman as Medusa, the gorgon with a coiff...",3.93438
2,m/0814255,Unknown Critic,False,FILMINK (Australia),Fresh,NaN,2010-02-09,With a top-notch cast and dazzling special eff...,3.93438
3,m/0814255,Ben McEachen,False,Sunday Mail (Australia),Fresh,3.5/5,2010-02-09,Whether audiences will get behind The Lightnin...,3.50000
4,m/0814255,Ethan Alter,True,Hollywood Reporter,Rotten,NaN,2010-02-10,What's really lacking in The Lightning Thief i...,3.93438


In [20]:
# Display the cleaned DataFrame to review the changes made during the cleaning process.
print(cleaned_critic_reviews_df)

        Rotten Tomatoes Link        Critic Name  Top Critic  \
0                  m/0814255    Andrew L. Urban       False   
1                  m/0814255      Louise Keller       False   
2                  m/0814255     Unknown Critic       False   
3                  m/0814255       Ben McEachen       False   
4                  m/0814255        Ethan Alter        True   
...                      ...                ...         ...   
1130012          m/zulu_dawn      Chuck O'Leary       False   
1130013          m/zulu_dawn          Ken Hanke       False   
1130014          m/zulu_dawn    Dennis Schwartz       False   
1130015          m/zulu_dawn  Christopher Lloyd       False   
1130016          m/zulu_dawn     Brent McKnight       False   

                          Publisher Name Review Type Review Score Review Date  \
0                         Urban Cinefile       Fresh          NaN  2010-02-06   
1                         Urban Cinefile       Fresh          NaN  2010-02-06   


In [21]:
# Calculate the Z-scores for 'Numeric Review Score.'
cleaned_critic_reviews_df["z_score"] = stats.zscore(cleaned_critic_reviews_df["Numeric Review Score"])

# Identify outliers (e.g., Z-score > 3 or < -3).
outliers = cleaned_critic_reviews_df[(cleaned_critic_reviews_df["z_score"] > 3) | (cleaned_critic_reviews_df["z_score"] < -3)]

# Display the outliers.
print(outliers)

           Rotten Tomatoes Link     Critic Name  Top Critic  \
262                   m/0878835   Philip Martin       False   
319      m/1000013-12_angry_men   Brian Webster       False   
349      m/1000013-12_angry_men     Dan Jardine       False   
607       m/1000123-310_to_yuma   Brian Webster       False   
662           m/1000224-accused   Jamie Gillies       False   
...                         ...             ...         ...   
1129317             m/zoolander    Mike DeWolfe       False   
1129573           m/zoolander_2  Piers Marchant       False   
1129906              m/zootopia   Philip Martin       False   
1129907              m/zootopia    Dan Lybarger       False   
1129983       m/zorba_the_greek     Dan Jardine       False   

                    Publisher Name Review Type Review Score Review Date  \
262      Arkansas Democrat-Gazette      Rotten       83/100  2010-07-03   
319                   Apollo Guide       Fresh       92/100  2001-04-18   
349               

- A total of 6005 rows were analyzed, and outliers were identified as those with Z-scores greater than 3 or less than -3. These include scores like 83.0, 92.0, and higher, indicating they are exceptionally high. 
- The outliers suggest unusually positive reviews and may represent specific critics or contexts contributing to these high scores.

- Significant outliers in the review scores, warrant further investigation to understand their implications. ns. 

In [22]:
# Check for unexpected values in categorical columns.
print(cleaned_critic_reviews_df["Critic Name"].value_counts())
print(cleaned_critic_reviews_df["Review Type"].value_counts())
print(cleaned_critic_reviews_df["Review Content"].value_counts())

Critic Name
Unknown Critic       7121
Emanuel Levy         7031
Dennis Schwartz      5821
Roger Ebert          5701
Brian Orndorf        5415
                     ... 
Joseph P. Kahn          1
Spencer Doar            1
Peter Momtchiloff       1
Henry Giardina          1
Will Heaven             1
Name: count, Length: 11109, dtype: int64
Review Type
Fresh     634968
Rotten    361502
Name: count, dtype: int64
Review Content
N/A                                                                                                                                                                                                                               53352
Parental Content Review                                                                                                                                                                                                             245
full review at Movies for the Masses                                                                          

**Critic Name:**
The dataset shows a significant concentration of reviews from a few critics, with “Unknown Critic” contributing **7121 reviews**. This indicates potential bias and over-reliance on certain voices. 

**Review Type:**
A notable predominance of “Fresh” reviews **634968** compared to “Rotten” reviews **361502** suggests a generally positive bias in the dataset. 

**Review Content:**
Many entries are marked as `N/A` **53352**, indicating missing review content. This raises concerns about data completeness.
Other entries vary widely, with some unique reviews only appearing once, pointing to a diverse range of opinions but limited utility due to high `N/A` counts.

The analysis highlights potential biases in critic representation and a positive skew in review types while also identifying significant gaps in the review content that could affect further analyses. 

In [23]:
# Check if 'Review Date' has any invalid dates or outliers
print(cleaned_critic_reviews_df["Review Date"].min(), cleaned_critic_reviews_df["Review Date"].max())

1800-01-01 00:00:00 2020-10-29 00:00:00


- The dataset shows a review date range from **January 1, 1800** to **October 29, 2020.** 
- The early date of 1800 is likely an invalid entry, as it predates modern film criticism and raises concerns about data accuracy. 
- This suggests potential data entry errors or inconsistencies that need to be addressed. 

# Movies DataFrame.

In [24]:
movies_df.head()

,rotten_tomatoes_link,movie_title,movie_info,critics_consensus,content_rating,genres,directors,authors,actors,original_release_date,...,production_company,tomatometer_status,tomatometer_rating,tomatometer_count,audience_status,audience_rating,audience_count,tomatometer_top_critics_count,tomatometer_fresh_critics_count,tomatometer_rotten_critics_count
0,m/0814255,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",Though it may seem like just another Harry Pot...,PG,"Action & Adventure, Comedy, Drama, Science Fic...",Chris Columbus,"Craig Titley, Chris Columbus, Rick Riordan","Logan Lerman, Brandon T. Jackson, Alexandra Da...",2010-02-12,...,20th Century Fox,Rotten,49.0,149.0,Spilled,53.0,254421.0,43,73,76
1,m/0878835,Please Give,Kate (Catherine Keener) and her husband Alex (...,Nicole Holofcener's newest might seem slight i...,R,Comedy,Nicole Holofcener,Nicole Holofcener,"Catherine Keener, Amanda Peet, Oliver Platt, R...",2010-04-30,...,Sony Pictures Classics,Certified-Fresh,87.0,142.0,Upright,64.0,11574.0,44,123,19
2,m/10,10,"A successful, middle-aged Hollywood songwriter...",Blake Edwards' bawdy comedy may not score a pe...,R,"Comedy, Romance",Blake Edwards,Blake Edwards,"Dudley Moore, Bo Derek, Julie Andrews, Robert ...",1979-10-05,...,Waner Bros.,Fresh,67.0,24.0,Spilled,53.0,14684.0,2,16,8
3,m/1000013-12_angry_men,12 Angry Men (Twelve Angry Men),Following the closing arguments in a murder tr...,Sidney Lumet's feature debut is a superbly wri...,NR,"Classics, Drama",Sidney Lumet,Reginald Rose,"Martin Balsam, John Fiedler, Lee J. Cobb, E.G....",1957-04-13,...,Criterion Collection,Certified-Fresh,100.0,54.0,Upright,97.0,105386.0,6,54,0
4,m/1000079-20000_leagues_under_the_sea,"20,000 Leagues Under The Sea","In 1866, Professor Pierre M. Aronnax (Paul Luk...","One of Disney's finest live-action adventures,...",G,"Action & Adventure, Drama, Kids & Family",Richard Fleischer,Earl Felton,"James Mason, Kirk Douglas, Paul Lukas, Peter L...",1954-01-01,...,Disney,Fresh,89.0,27.0,Upright,74.0,68918.0,5,24,3


In [25]:
# Get the original dimensions of the movies DataFrame.
movies_df.shape

(17712, 22)

In [26]:
# Display the data types of each column in the DataFrame to understand the structure of the data.
movies_df.dtypes

rotten_tomatoes_link                 object
movie_title                          object
movie_info                           object
critics_consensus                    object
content_rating                       object
genres                               object
directors                            object
authors                              object
actors                               object
original_release_date                object
streaming_release_date               object
runtime                             float64
production_company                   object
tomatometer_status                   object
tomatometer_rating                  float64
tomatometer_count                   float64
audience_status                      object
audience_rating                     float64
audience_count                      float64
tomatometer_top_critics_count         int64
tomatometer_fresh_critics_count       int64
tomatometer_rotten_critics_count      int64
dtype: object

In [27]:
# STEP 1.
# Create a new DataFrame cleaned_movies_df from movies_df.
cleaned_movies_df = movies_df.copy()

In [28]:
# STEP 2.
# Format Headers: Removing underscores and capitalize the first letter of each word.
cleaned_movies_df.columns = cleaned_movies_df.columns.str.replace("_", " ").str.title()

In [29]:
# Display first few rows of cleaned_critic_reviews_df to verify change to headers.
cleaned_movies_df.head()

,Rotten Tomatoes Link,Movie Title,Movie Info,Critics Consensus,Content Rating,Genres,Directors,Authors,Actors,Original Release Date,...,Production Company,Tomatometer Status,Tomatometer Rating,Tomatometer Count,Audience Status,Audience Rating,Audience Count,Tomatometer Top Critics Count,Tomatometer Fresh Critics Count,Tomatometer Rotten Critics Count
0,m/0814255,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",Though it may seem like just another Harry Pot...,PG,"Action & Adventure, Comedy, Drama, Science Fic...",Chris Columbus,"Craig Titley, Chris Columbus, Rick Riordan","Logan Lerman, Brandon T. Jackson, Alexandra Da...",2010-02-12,...,20th Century Fox,Rotten,49.0,149.0,Spilled,53.0,254421.0,43,73,76
1,m/0878835,Please Give,Kate (Catherine Keener) and her husband Alex (...,Nicole Holofcener's newest might seem slight i...,R,Comedy,Nicole Holofcener,Nicole Holofcener,"Catherine Keener, Amanda Peet, Oliver Platt, R...",2010-04-30,...,Sony Pictures Classics,Certified-Fresh,87.0,142.0,Upright,64.0,11574.0,44,123,19
2,m/10,10,"A successful, middle-aged Hollywood songwriter...",Blake Edwards' bawdy comedy may not score a pe...,R,"Comedy, Romance",Blake Edwards,Blake Edwards,"Dudley Moore, Bo Derek, Julie Andrews, Robert ...",1979-10-05,...,Waner Bros.,Fresh,67.0,24.0,Spilled,53.0,14684.0,2,16,8
3,m/1000013-12_angry_men,12 Angry Men (Twelve Angry Men),Following the closing arguments in a murder tr...,Sidney Lumet's feature debut is a superbly wri...,NR,"Classics, Drama",Sidney Lumet,Reginald Rose,"Martin Balsam, John Fiedler, Lee J. Cobb, E.G....",1957-04-13,...,Criterion Collection,Certified-Fresh,100.0,54.0,Upright,97.0,105386.0,6,54,0
4,m/1000079-20000_leagues_under_the_sea,"20,000 Leagues Under The Sea","In 1866, Professor Pierre M. Aronnax (Paul Luk...","One of Disney's finest live-action adventures,...",G,"Action & Adventure, Drama, Kids & Family",Richard Fleischer,Earl Felton,"James Mason, Kirk Douglas, Paul Lukas, Peter L...",1954-01-01,...,Disney,Fresh,89.0,27.0,Upright,74.0,68918.0,5,24,3


In [30]:
# Calculate the total number of missing values in each column of the "cleaned_movies_df."
movies_missing_values = cleaned_movies_df.isnull().sum()
print(movies_missing_values)

Rotten Tomatoes Link                   0
Movie Title                            0
Movie Info                           321
Critics Consensus                   8578
Content Rating                         0
Genres                                19
Directors                            194
Authors                             1542
Actors                               352
Original Release Date               1166
Streaming Release Date               384
Runtime                              314
Production Company                   499
Tomatometer Status                    44
Tomatometer Rating                    44
Tomatometer Count                     44
Audience Status                      448
Audience Rating                      296
Audience Count                       297
Tomatometer Top Critics Count          0
Tomatometer Fresh Critics Count        0
Tomatometer Rotten Critics Count       0
dtype: int64


In [31]:
# STEPS 3 - 11.
# Fill missing values in categorical (object) columns with the appropriate placeholder.
cleaned_movies_df["Movie Info"] = cleaned_movies_df["Movie Info"].fillna("Unknown Info")
cleaned_movies_df["Critics Consensus"] = cleaned_movies_df["Critics Consensus"].fillna("No Consensus")
cleaned_movies_df["Genres"] = cleaned_movies_df["Genres"].fillna("Unknown Genre")
cleaned_movies_df["Directors"] = cleaned_movies_df["Directors"].fillna("Unknown Director")
cleaned_movies_df["Authors"] = cleaned_movies_df["Authors"].fillna("Unknown Author")
cleaned_movies_df["Actors"] = cleaned_movies_df["Actors"].fillna("Unknown Actors")
cleaned_movies_df["Production Company"] = cleaned_movies_df["Production Company"].fillna("Unknown Production Company")
cleaned_movies_df["Tomatometer Status"] = cleaned_movies_df["Tomatometer Status"].fillna("Unknown Status")
cleaned_movies_df["Audience Status"] = cleaned_movies_df["Audience Status"].fillna("Unknown Audience Status")

In [32]:
# STEPS 12 - 15.
# Fill missing numerical (float64) columns with the median.
cleaned_movies_df["Tomatometer Rating"] = cleaned_movies_df["Tomatometer Rating"].fillna(cleaned_movies_df["Tomatometer Rating"].median())
cleaned_movies_df["Tomatometer Count"] = cleaned_movies_df["Tomatometer Count"].fillna(cleaned_movies_df["Tomatometer Count"].median())
cleaned_movies_df["Audience Rating"] = cleaned_movies_df["Audience Rating"].fillna(cleaned_movies_df["Audience Rating"].median())
cleaned_movies_df["Audience Count"] = cleaned_movies_df["Audience Count"].fillna(cleaned_movies_df["Audience Count"].median())

In [33]:
# STEPS 16 - 17.
# Convert "Original Release Date" and "Streaming Release Date" to datetime format
cleaned_movies_df["Original Release Date"] = pd.to_datetime(cleaned_movies_df["Original Release Date"], errors = "coerce")
cleaned_movies_df["Streaming Release Date"] = pd.to_datetime(cleaned_movies_df["Streaming Release Date"], errors = "coerce")

In [34]:
# STEPS 18 - 19.
# Fill missing "Original Release Date" and "Streaming Release Date" with a placeholder date
cleaned_movies_df["Original Release Date"] = cleaned_movies_df["Original Release Date"].fillna(pd.to_datetime("1800-01-01"))
cleaned_movies_df["Streaming Release Date"] = cleaned_movies_df["Streaming Release Date"].fillna(pd.to_datetime("1800-01-01"))

In [35]:
# Step 20.
# Fill missing values in 'Runtime' with 0 as a placeholder.
cleaned_movies_df["Runtime"] = cleaned_movies_df["Runtime"].fillna(0)

In [36]:
# Recheck missing values in each column of the "cleaned_movies_df."
movies_missing_values = cleaned_movies_df.isnull().sum()
print(movies_missing_values)

Rotten Tomatoes Link                0
Movie Title                         0
Movie Info                          0
Critics Consensus                   0
Content Rating                      0
Genres                              0
Directors                           0
Authors                             0
Actors                              0
Original Release Date               0
Streaming Release Date              0
Runtime                             0
Production Company                  0
Tomatometer Status                  0
Tomatometer Rating                  0
Tomatometer Count                   0
Audience Status                     0
Audience Rating                     0
Audience Count                      0
Tomatometer Top Critics Count       0
Tomatometer Fresh Critics Count     0
Tomatometer Rotten Critics Count    0
dtype: int64


In [37]:
# Check unique values 'Tomatometer Status' and 'Audience Status' to identify distinct review categories.
print(cleaned_movies_df['Tomatometer Status'].unique())
print(cleaned_movies_df['Audience Status'].unique())


['Rotten' 'Certified-Fresh' 'Fresh' 'Unknown Status']
['Spilled' 'Upright' 'Unknown Audience Status']


In [38]:
# Check for duplicates based on "Rotten Tomatoes Link"
duplicates_rotten_tomatoes = cleaned_movies_df.duplicated(subset = "Rotten Tomatoes Link", keep = False)

# Check for duplicates based on "Movie Title".
duplicates_movie_title = cleaned_movies_df.duplicated(subset = "Movie Title", keep = False)

# Display duplicate rows based on "Rotten Tomatoes Link" and "Movie Title"
print(cleaned_movies_df[duplicates_rotten_tomatoes])
print(cleaned_movies_df[duplicates_movie_title])

Empty DataFrame
Columns: [Rotten Tomatoes Link, Movie Title, Movie Info, Critics Consensus, Content Rating, Genres, Directors, Authors, Actors, Original Release Date, Streaming Release Date, Runtime, Production Company, Tomatometer Status, Tomatometer Rating, Tomatometer Count, Audience Status, Audience Rating, Audience Count, Tomatometer Top Critics Count, Tomatometer Fresh Critics Count, Tomatometer Rotten Critics Count]
Index: []

[0 rows x 22 columns]
            Rotten Tomatoes Link        Movie Title  \
7          m/1000123-310_to_yuma       3:10 to Yuma   
10         m/10002114-dark_water         Dark Water   
17           m/10003276-criminal           Criminal   
22           m/10003925-dead_end           Dead End   
28     m/10004288-running_scared     Running Scared   
...                          ...                ...   
17496        m/wonder_woman_2017       Wonder Woman   
17553        m/wuthering_heights  Wuthering Heights   
17554   m/wuthering_heights_2011  Wuthering H

In [39]:
# Steps 21 - 22.
# Remove duplicates based on "Rotten Tomatoes Link" while keeping the first occurrence.
cleaned_movies_df.drop_duplicates(subset = "Rotten Tomatoes Link", keep = "first", inplace = True)

# Remove duplicates based on "Movie Title"
cleaned_movies_df.drop_duplicates(subset = "Movie Title", keep = "first", inplace = True)

In [40]:
# Check for remaining duplicates in 'rotten_tomatoes_link' and 'movie_title'
print(f"Remaining duplicates in 'Rotten Tomatoes Link': {cleaned_movies_df.duplicated(subset = 'Rotten Tomatoes Link').sum()}")
print(f"Remaining duplicates in 'Movie Title': {cleaned_movies_df.duplicated(subset = 'Movie Title').sum()}")

Remaining duplicates in 'Rotten Tomatoes Link': 0
Remaining duplicates in 'Movie Title': 0


In [41]:
# Get the shape of cleaned_movies_df.
cleaned_movies_df.shape

(17106, 22)

In [42]:
# Display cleaned_movies_df types of each column in the DataFrame.
cleaned_movies_df.dtypes

Rotten Tomatoes Link                        object
Movie Title                                 object
Movie Info                                  object
Critics Consensus                           object
Content Rating                              object
Genres                                      object
Directors                                   object
Authors                                     object
Actors                                      object
Original Release Date               datetime64[ns]
Streaming Release Date              datetime64[ns]
Runtime                                    float64
Production Company                          object
Tomatometer Status                          object
Tomatometer Rating                         float64
Tomatometer Count                          float64
Audience Status                             object
Audience Rating                            float64
Audience Count                             float64
Tomatometer Top Critics Count  

In [43]:
# Display first few rows of the cleaned_movies_df.
cleaned_movies_df.head()

,Rotten Tomatoes Link,Movie Title,Movie Info,Critics Consensus,Content Rating,Genres,Directors,Authors,Actors,Original Release Date,...,Production Company,Tomatometer Status,Tomatometer Rating,Tomatometer Count,Audience Status,Audience Rating,Audience Count,Tomatometer Top Critics Count,Tomatometer Fresh Critics Count,Tomatometer Rotten Critics Count
0,m/0814255,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",Though it may seem like just another Harry Pot...,PG,"Action & Adventure, Comedy, Drama, Science Fic...",Chris Columbus,"Craig Titley, Chris Columbus, Rick Riordan","Logan Lerman, Brandon T. Jackson, Alexandra Da...",2010-02-12,...,20th Century Fox,Rotten,49.0,149.0,Spilled,53.0,254421.0,43,73,76
1,m/0878835,Please Give,Kate (Catherine Keener) and her husband Alex (...,Nicole Holofcener's newest might seem slight i...,R,Comedy,Nicole Holofcener,Nicole Holofcener,"Catherine Keener, Amanda Peet, Oliver Platt, R...",2010-04-30,...,Sony Pictures Classics,Certified-Fresh,87.0,142.0,Upright,64.0,11574.0,44,123,19
2,m/10,10,"A successful, middle-aged Hollywood songwriter...",Blake Edwards' bawdy comedy may not score a pe...,R,"Comedy, Romance",Blake Edwards,Blake Edwards,"Dudley Moore, Bo Derek, Julie Andrews, Robert ...",1979-10-05,...,Waner Bros.,Fresh,67.0,24.0,Spilled,53.0,14684.0,2,16,8
3,m/1000013-12_angry_men,12 Angry Men (Twelve Angry Men),Following the closing arguments in a murder tr...,Sidney Lumet's feature debut is a superbly wri...,NR,"Classics, Drama",Sidney Lumet,Reginald Rose,"Martin Balsam, John Fiedler, Lee J. Cobb, E.G....",1957-04-13,...,Criterion Collection,Certified-Fresh,100.0,54.0,Upright,97.0,105386.0,6,54,0
4,m/1000079-20000_leagues_under_the_sea,"20,000 Leagues Under The Sea","In 1866, Professor Pierre M. Aronnax (Paul Luk...","One of Disney's finest live-action adventures,...",G,"Action & Adventure, Drama, Kids & Family",Richard Fleischer,Earl Felton,"James Mason, Kirk Douglas, Paul Lukas, Peter L...",1954-01-01,...,Disney,Fresh,89.0,27.0,Upright,74.0,68918.0,5,24,3


In [44]:
# Display the cleaned DataFrame to review the changes made during the cleaning process.
print(cleaned_movies_df)

                        Rotten Tomatoes Link  \
0                                  m/0814255   
1                                  m/0878835   
2                                       m/10   
3                     m/1000013-12_angry_men   
4      m/1000079-20000_leagues_under_the_sea   
...                                      ...   
17707                            m/zoot_suit   
17708                             m/zootopia   
17709                      m/zorba_the_greek   
17710                                 m/zulu   
17711                            m/zulu_dawn   

                                             Movie Title  \
0      Percy Jackson & the Olympians: The Lightning T...   
1                                            Please Give   
2                                                     10   
3                        12 Angry Men (Twelve Angry Men)   
4                           20,000 Leagues Under The Sea   
...                                                  ...   
177

In [45]:
# Defining a list of numeric (float64) columns for analysis.
numeric_columns = ["Tomatometer Rating", "Audience Rating", "Tomatometer Count", "Audience Count"]

# Loop through each column to identify outliers using Z-score
for column in numeric_columns:
    # Calculate the Z-scores
    cleaned_movies_df[f'z_score_{column}'] = stats.zscore(cleaned_movies_df[column])
    
    # Identify outliers with Z-scores > 3 or < -3
    outliers = cleaned_movies_df[(cleaned_movies_df[f'z_score_{column}'] > 3) | (cleaned_movies_df[f'z_score_{column}'] < -3)]
    
    print(f"Outliers in {column}:")
    print(outliers[[column, f'z_score_{column}']])

Outliers in Tomatometer Rating:
Empty DataFrame
Columns: [Tomatometer Rating, z_score_Tomatometer Rating]
Index: []
Outliers in Audience Rating:
Empty DataFrame
Columns: [Audience Rating, z_score_Audience Rating]
Index: []
Outliers in Tomatometer Count:
       Tomatometer Count  z_score_Tomatometer Count
169                278.0                   3.300460
1306               309.0                   3.761389
1623               259.0                   3.017955
1693               258.0                   3.003086
1837               263.0                   3.077430
...                  ...                        ...
17560              295.0                   3.553227
17600              349.0                   4.356136
17628              277.0                   3.285591
17684              298.0                   3.597833
17708              291.0                   3.493752

[332 rows x 2 columns]
Outliers in Audience Count:
       Audience Count  z_score_Audience Count
701        33403994.0   

**Tomatometer Rating:** No outliers were found, indicating a normal distribution of ratings. 

**Audience Rating:** No outliers were identified, suggesting consistent ratings across the dataset. 

**Tomatometer Count**: Several outliers were detected, ranging from approximately 3.0 to 4.35, indicating that some films received significantly more critic reviews, likely due to popularity or acclaim. 

**Audience Count:** Multiple outliers were found, with extremely high (exceeding 18) review counts for certain films, reflecting strong audience engagement. 

In [46]:
# Check for unique values in categorical columns
print(cleaned_movies_df['Genres'].value_counts())
print(cleaned_movies_df['Directors'].value_counts())

Genres
Drama                                                                                  1828
Comedy                                                                                 1233
Comedy, Drama                                                                           843
Drama, Mystery & Suspense                                                               702
Art House & International, Drama                                                        573
                                                                                       ... 
Art House & International, Classics, Cult Movies, Horror, Science Fiction & Fantasy       1
Action & Adventure, Cult Movies, Drama, Science Fiction & Fantasy                         1
Art House & International, Documentary, Sports & Fitness                                  1
Action & Adventure, Drama, Mystery & Suspense, Special Interest                           1
Action & Adventure, Drama, Horror, Kids & Family, Mystery & Suspense     

**Genres:** 
- Drama (1828) and Comedy (1233) dominate, showing a strong audience preference. 
- Comedy and Drama (843) indicates a blend of these styles. 
- Genres like Drama, Mystery & Suspense (702) and Art House & International, Drama (573) highlight varied storytelling approaches. 
- There are 1090 unique genre combinations in the dataset. 

**Directors**: 
- Unknown Director (189) points to many films lacking prominent names, often from independent projects. 
- Woody Allen and Clint Eastwood (36 each) are significant figures in the dataset. 
- Alfred Hitchcock (34) reflects the legacy of classic directors. 
- The dataset includes 8748 unique directors, showcasing a wide range of talent. 

In [47]:
# Check for missing values in the dataset.
missing_values = cleaned_movies_df.isnull().sum()
print("Missing values in each column:")
print(missing_values)

# Check for unusual zero values in important columns.
invalid_runtime = cleaned_movies_df[cleaned_movies_df['Runtime'] == 0]
print("Movies with runtime of 0 (possible invalid data):")
print(invalid_runtime[['Movie Title', 'Runtime']])

Missing values in each column:
Rotten Tomatoes Link                0
Movie Title                         0
Movie Info                          0
Critics Consensus                   0
Content Rating                      0
Genres                              0
Directors                           0
Authors                             0
Actors                              0
Original Release Date               0
Streaming Release Date              0
Runtime                             0
Production Company                  0
Tomatometer Status                  0
Tomatometer Rating                  0
Tomatometer Count                   0
Audience Status                     0
Audience Rating                     0
Audience Count                      0
Tomatometer Top Critics Count       0
Tomatometer Fresh Critics Count     0
Tomatometer Rotten Critics Count    0
z_score_Tomatometer Rating          0
z_score_Audience Rating             0
z_score_Tomatometer Count           0
z_score_Audience Co

No missing values are in any column of the “cleaned_movies_df" DataFrame, indicating complete and reliable data for analysis. 

There are 302 movies with a runtime of 0, which is likely an error, as all films should have a defined duration. This raises concerns about data accuracy. 

Multiple entries with a runtime of 0 suggest potential data entry issues that require further investigation. 

Correct or exclude the zero-runtime entries to ensure data integrity. 

# Human readable dataset after all transformations for "cleaned_critic_reviews_df."

In [48]:
# Display the cleaned cleaned_critic_reviews_df DataFrame to review the changes made during the cleaning process.
print(cleaned_critic_reviews_df)

        Rotten Tomatoes Link        Critic Name  Top Critic  \
0                  m/0814255    Andrew L. Urban       False   
1                  m/0814255      Louise Keller       False   
2                  m/0814255     Unknown Critic       False   
3                  m/0814255       Ben McEachen       False   
4                  m/0814255        Ethan Alter        True   
...                      ...                ...         ...   
1130012          m/zulu_dawn      Chuck O'Leary       False   
1130013          m/zulu_dawn          Ken Hanke       False   
1130014          m/zulu_dawn    Dennis Schwartz       False   
1130015          m/zulu_dawn  Christopher Lloyd       False   
1130016          m/zulu_dawn     Brent McKnight       False   

                          Publisher Name Review Type Review Score Review Date  \
0                         Urban Cinefile       Fresh          NaN  2010-02-06   
1                         Urban Cinefile       Fresh          NaN  2010-02-06   


# Human readable dataset after all transformations for "cleaned_movies_df."

In [49]:
# Display the cleaned_movies_df DataFrame to review the changes made during the cleaning process.
print(cleaned_movies_df)

                        Rotten Tomatoes Link  \
0                                  m/0814255   
1                                  m/0878835   
2                                       m/10   
3                     m/1000013-12_angry_men   
4      m/1000079-20000_leagues_under_the_sea   
...                                      ...   
17707                            m/zoot_suit   
17708                             m/zootopia   
17709                      m/zorba_the_greek   
17710                                 m/zulu   
17711                            m/zulu_dawn   

                                             Movie Title  \
0      Percy Jackson & the Olympians: The Lightning T...   
1                                            Please Give   
2                                                     10   
3                        12 Angry Men (Twelve Angry Men)   
4                           20,000 Leagues Under The Sea   
...                                                  ...   
177

# Ethical implications of data wrangling specific to my datasources.

    In the process of cleaning and transforming the “leaned_critic_reviews_df” and “cleaned_movies_df” datasets, several modifications were made, including handling missing values, standardizing categorical columns, filling null values with placeholders, and identifying/removing duplicates. These changes could introduce risks related to data integrity, such as altering the intended meaning of reviews or movie ratings, especially when assumptions like filling missing scores or dates were made. Since movie data and reviews are often public, fewer legal concerns may exist. Still, it is critical to comply with privacy and data protection regulations, especially if the personal information of critics is involved. Data credibility is sourced from established platforms like Rotten Tomatoes, but ethical acquisition and transparency in data handling remain important. To mitigate potential ethical concerns, it is essential to document all transformations, ensure that assumptions do not bias the analysis, and handle public reviews and movie metadata in a way that respects the source and original context. 

In [50]:
cleaned_critic_reviews_df.head()

,Rotten Tomatoes Link,Critic Name,Top Critic,Publisher Name,Review Type,Review Score,Review Date,Review Content,Numeric Review Score,z_score
0,m/0814255,Andrew L. Urban,False,Urban Cinefile,Fresh,NaN,2010-02-06,A fantasy adventure that fuses Greek mythology...,3.93438,0.000392
1,m/0814255,Louise Keller,False,Urban Cinefile,Fresh,NaN,2010-02-06,"Uma Thurman as Medusa, the gorgon with a coiff...",3.93438,0.000392
2,m/0814255,Unknown Critic,False,FILMINK (Australia),Fresh,NaN,2010-02-09,With a top-notch cast and dazzling special eff...,3.93438,0.000392
3,m/0814255,Ben McEachen,False,Sunday Mail (Australia),Fresh,3.5/5,2010-02-09,Whether audiences will get behind The Lightnin...,3.50000,-0.077095
4,m/0814255,Ethan Alter,True,Hollywood Reporter,Rotten,NaN,2010-02-10,What's really lacking in The Lightning Thief i...,3.93438,0.000392


In [51]:
cleaned_movies_df.head()

,Rotten Tomatoes Link,Movie Title,Movie Info,Critics Consensus,Content Rating,Genres,Directors,Authors,Actors,Original Release Date,...,Audience Status,Audience Rating,Audience Count,Tomatometer Top Critics Count,Tomatometer Fresh Critics Count,Tomatometer Rotten Critics Count,z_score_Tomatometer Rating,z_score_Audience Rating,z_score_Tomatometer Count,z_score_Audience Count
0,m/0814255,Percy Jackson & the Olympians: The Lightning T...,"Always trouble-prone, the life of teenager Per...",Though it may seem like just another Harry Pot...,PG,"Action & Adventure, Comedy, Drama, Science Fic...",Chris Columbus,"Craig Titley, Chris Columbus, Rick Riordan","Logan Lerman, Brandon T. Jackson, Alexandra Da...",2010-02-12,...,Spilled,53.0,254421.0,43,73,76,-0.422902,-0.378467,1.382400,0.073007
1,m/0878835,Please Give,Kate (Catherine Keener) and her husband Alex (...,Nicole Holofcener's newest might seem slight i...,R,Comedy,Nicole Holofcener,Nicole Holofcener,"Catherine Keener, Amanda Peet, Oliver Platt, R...",2010-04-30,...,Upright,64.0,11574.0,44,123,19,0.914255,0.161389,1.278319,-0.072841
2,m/10,10,"A successful, middle-aged Hollywood songwriter...",Blake Edwards' bawdy comedy may not score a pe...,R,"Comedy, Romance",Blake Edwards,Blake Edwards,"Dudley Moore, Bo Derek, Julie Andrews, Robert ...",1979-10-05,...,Spilled,53.0,14684.0,2,16,8,0.210488,-0.378467,-0.476184,-0.070973
3,m/1000013-12_angry_men,12 Angry Men (Twelve Angry Men),Following the closing arguments in a murder tr...,Sidney Lumet's feature debut is a superbly wri...,NR,"Classics, Drama",Sidney Lumet,Reginald Rose,"Martin Balsam, John Fiedler, Lee J. Cobb, E.G....",1957-04-13,...,Upright,97.0,105386.0,6,54,0,1.371703,1.780959,-0.030124,-0.016500
4,m/1000079-20000_leagues_under_the_sea,"20,000 Leagues Under The Sea","In 1866, Professor Pierre M. Aronnax (Paul Luk...","One of Disney's finest live-action adventures,...",G,"Action & Adventure, Drama, Kids & Family",Richard Fleischer,Earl Felton,"James Mason, Kirk Douglas, Paul Lukas, Peter L...",1954-01-01,...,Upright,74.0,68918.0,5,24,3,0.984632,0.652168,-0.431578,-0.038402


In [52]:
cleaned_critic_reviews_df.to_csv("rt_cleaned_critic_reviews_df.csv", index=False)

In [53]:
cleaned_movies_df.to_csv("rt_cleaned_movies_df.csv", index=False)